In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from  collections import OrderedDict
import torch
import pickle
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

def get_cmap(n, name='hsv'):
    '''Returns a function that maps each index in 0, 1, ..., n-1 to a distinct 
    RGB color; the keyword argument name must be a standard mpl colormap name.'''
    return plt.cm.get_cmap(name, n)

In [ ]:
split="train"
nft2emb_unaware = pickle.load( open(f"../embeddings/nft2emb_dummyViT-{split}.pkl", 'rb'))
TOP_K=10


In [ ]:
emb_keys=[nft for nft in nft2emb_unaware]
colls=[eval(nft)[0] for nft in nft2emb_unaware]
colls={k:v for k,v in sorted({coll:count for coll,count in zip(*np.unique(colls,return_counts=True))}.items(),key=lambda x:-x[1])}
series=pd.Series(colls).iloc[:TOP_K]
series=series.to_dict()
coll2id={k:i for i,k in enumerate(series)}
id2coll={v:k for k,v in coll2id.items()}
coll2id


In [ ]:
N=190000
embs=torch.cat([emb.unsqueeze(0) for emb in nft2emb_unaware.values()]).numpy()[:N]
colls=[coll2id.get(eval(nft)[0],-1) for nft in nft2emb_unaware][:N]

std_scaler=StandardScaler()
std_scaler.fit(embs)

In [ ]:
metric="cosine"
standard_embedding = Pipeline(
        [('scaler', StandardScaler()), 
        ('umap', umap.UMAP(random_state=42, n_neighbors=30,metric=metric,n_jobs=24))]
        ).fit_transform(embs)

In [ ]:
import random 
number_of_colors = TOP_K+1

hexadecimal_alphabets = '0123456789ABCDEF'

color = ["#" + ''.join([random.choice(hexadecimal_alphabets) for j in range(6)]) for i in range(number_of_colors)]

In [ ]:
import matplotlib as mpl
mpl.colors.ListedColormap(color)

In [ ]:
from matplotlib.cm import ScalarMappable
fig, ax = plt.subplots(figsize = (60,30))
# ax.set_facecolor('black')
classes=[]
for coll in np.unique(colls):
    emb_x=standard_embedding[:, 0][colls==coll]
    emb_y=standard_embedding[:, 1][colls==coll]
    coll_name=id2coll.get(coll,"Other")
    scatter = ax.scatter(emb_x, emb_y, c=color[coll],s=1.2,label=coll_name)
    classes.append(coll_name)


# cbar = plt.colorbar(mpl.colors.ListedColormap(color))
# tick_locs = (np.arange(11) + 0.5)*(11-1)/11
# cbar.set_ticks(tick_locs)

plt.legend( prop={'size': 30},markerscale=16, )
plt.savefig("./umap.pdf")
plt.show()